# AI Documentary Image Generator

Generate consistent AI images for the AI Documentary pipeline.

**Free GPU:** Colab/Kaggle T4 (16GB VRAM).

## Setup
1. Upload your scenario markdown file to Colab.
2. Run all cells.
3. Download the generated images folder.
4. Place images in `data/ai_doc/<project>/images/`.


In [ ]:
# Install dependencies
!pip install diffusers transformers accelerate safetensors torch --quiet


## Music selection

Choose your background music before generating. Options:

| Track | Mood | BPM | Use case |
|---|---|---:|---|
| `tense_ambient.wav` | Tense, cynical, ambient | 68 | Documentary, philosophy |
| `dark_industrial_loop.wav` | Aggressive, driving | 134 | Drama, challenges, shorts |
| `generated_phonk_loop.wav` | Phonk, energetic | 132 | Shorts, drama |

Upload your track to `data/music/` or set `MUSIC_PATH` below.


In [ ]:
# Configuration
import os

SCENARIO_PATH = 'machiavelli_001.md'  # upload your scenario
OUTPUT_DIR = '/content/images'
MUSIC_PATH = '/content/music.wav'  # upload your track
MODEL_ID = 'runwayml/stable-diffusion-v1-5'  # or 'stabilityai/stable-diffusion-xl-base-1.0'
STYLE_SUFFIX = 'old master oil painting, dark moody chiaroscuro, cinematic composition, umber and gold palette'
SEED = 42
WIDTH = 1024
HEIGHT = 576  # 16:9
STEPS = 30
GUIDANCE = 7.5

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Parse scenario markdown
import re

def parse_scenario(path):
    text = open(path, encoding='utf-8').read()
    # Parse front matter
    meta = {}
    body = text
    m = re.match(r'^---\s*\n(.*?)\n---\s*\n', text, re.DOTALL)
    if m:
        body = text[m.end():]
        for line in m.group(1).splitlines():
            if ':' in line:
                key, val = line.split(':', 1)
                meta[key.strip().lower()] = val.strip()
    
    segments = []
    current = {}
    for line in body.splitlines():
        line = line.strip()
        m = re.match(r'^##\s+(?:Segment\s*)?\d*\s*[-—:]?\s*(.+?)$', line)
        if m:
            if current and current.get('prompt'):
                segments.append(current)
            current = {'title': m.group(1).strip(), 'prompt': '', 'voice': '', 'subtitle': ''}
            continue
        m = re.match(r'^(image|prompt|voice|subtitle|music)\s*:\s*(.+?)$', line)
        if m:
            key = m.group(1).lower()
            val = m.group(2).strip().strip('"').strip("'")
            if key == 'image':
                current['image'] = val
            elif key == 'prompt':
                current['prompt'] = val
            elif key == 'voice':
                current['voice'] = val
            elif key == 'subtitle':
                current['subtitle'] = val
    if current and current.get('prompt'):
        segments.append(current)
    return meta, segments

meta, segments = parse_scenario(SCENARIO_PATH)
print(f'Found {len(segments)} segments')
for i, s in enumerate(segments):
    print(f'  {i+1}. {s["title"]}: {s["prompt"][:60]}...')


In [ ]:
# Load Stable Diffusion
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()
print('Model loaded')


In [ ]:
# Generate images
import torch, os
from PIL import Image

generator = torch.Generator(device='cuda').manual_seed(SEED)

for i, seg in enumerate(segments):
    prompt = seg['prompt'] + ', ' + STYLE_SUFFIX
    out_path = os.path.join(OUTPUT_DIR, f'{i:03d}.png')
    if os.path.exists(out_path):
        print(f'[{i+1}/{len(segments)}] skip (exists): {out_path}')
        continue
    print(f'[{i+1}/{len(segments)}] generating: {seg["title"]}')
    image = pipe(
        prompt=prompt,
        negative_prompt='text, watermark, signature, low quality, blurry, deformed',
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        generator=generator,
    ).images[0]
    image.save(out_path)
    print(f'  saved: {out_path}')

print(f'\nDone! {len(segments)} images in {OUTPUT_DIR}')


In [ ]:
# Download images as zip
import shutil, zipfile, os
from google.colab import files

zip_path = '/content/images.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith(('.png', '.jpg', '.jpeg')):
            zf.write(os.path.join(OUTPUT_DIR, f), f)
print(f'Zip created: {zip_path} ({len(os.listdir(OUTPUT_DIR))} files)')
files.download(zip_path)

# Also download the final video (if assembled in Colab)
VIDEO_PATH = '/content/output.mp4'
if os.path.exists(VIDEO_PATH):
    files.download(VIDEO_PATH)
    print(f'Downloading: {VIDEO_PATH}')
else:
    print(f'No final video at {VIDEO_PATH} — assemble locally with shorts-factory')
